# 🌿 REAL-TIME PLANT DISEASE DIAGNOSIS - GOOGLE COLAB TRAINING PIPELINE
### Đề tài: Chẩn đoán bệnh và sức khỏe cây cảnh khuôn viên ĐH Phenikaa
**Mô hình huấn luyện**: `DSCNet` (Siêu nhẹ), `ResNet18` (Baseline), `SE-ResNet18` (Attention)
**Số epochs**: 100 epochs mỗi mô hình | **Device**: GPU (Tesla T4)

## Bước 1: Kiểm tra cấu hình GPU Colab (Tesla T4)
*(Vào menu: Runtime -> Change runtime type -> Chọn T4 GPU -> Bấm Save)*

In [ ]:
!nvidia-smi

## Bước 2: Giải nén mã nguồn và bộ dữ liệu PCOPD
*(Hãy tải file `plant_disease_colab.zip` lên mục Files ở thanh bên trái của Colab, hoặc kéo thả vào)*

In [ ]:
import os
if os.path.exists('/content/plant_disease_colab.zip'):
    !unzip -q /content/plant_disease_colab.zip -d /content/project
    %cd /content/project
    print("Đã giải nén thành công mã nguồn và dữ liệu!")
elif os.path.exists('/content/drive/MyDrive/plant_disease_colab.zip'):
    !unzip -q /content/drive/MyDrive/plant_disease_colab.zip -d /content/project
    %cd /content/project
    print("Đã giải nén thành công mã nguồn từ Google Drive!")
else:
    print("Vui lòng tải file 'plant_disease_colab.zip' lên thư mục /content/ trước!")

## Bước 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -r requirements.txt

## Bước 4: Huấn luyện Mô hình 1: DSCNet (Mạng tự đề xuất siêu nhẹ ~2.68M params) - 100 Epochs
*Có hỗ trợ Mixed Precision (AMP fp16) trên GPU, tốc độ cực nhanh (~6-8 phút xong 100 epochs)*

In [ ]:
!python src/training/train.py --model dscnet --epochs 100 --batch_size 32

## Bước 5: Huấn luyện Mô hình 2: ResNet18 (Baseline đối chứng ~11.18M params) - 100 Epochs

In [ ]:
!python src/training/train.py --model resnet18 --epochs 100 --batch_size 32

## Bước 6: Huấn luyện Mô hình 3: SE-ResNet18 (Attention ~11.27M params) - 100 Epochs

In [ ]:
!python src/training/train.py --model se_resnet18 --epochs 100 --batch_size 32

## Bước 7: Hiển thị Biểu đồ Huấn luyện (Learning Curves) của 3 mô hình

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

models = ["dscnet", "resnet18", "se_resnet18"]
for m in models:
    p = f"weights/{m}_learning_curves.png"
    if os.path.exists(p):
        plt.figure(figsize=(15, 4))
        img = mpimg.imread(p)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Learning Curves for {m.upper()}")
        plt.show()

## Bước 8: Xuất mô hình DSCNet sang định dạng ONNX phục vụ Real-Time App

In [ ]:
!python src/export/export_model.py

## Bước 9: Nén toàn bộ trọng số huấn luyện và biểu đồ để tải về máy

In [ ]:
!zip -r trained_weights_and_results.zip weights/
from google.colab import files
files.download('trained_weights_and_results.zip')